# 02 — Carbon Accounting
**Levante Ferries · portfolio ficticio · 2026**

Ejecutar desde la raíz del proyecto o desde `notebooks`. Cada tabla se calcula sobre CSV regenerables.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
import pandas as pd
import numpy as np
from model import assess,aggregate,calculate,optimize,FUELEU_LIMIT
DATA=ROOT/'data'; OUT=ROOT/'outputs'
voyages=pd.read_csv(DATA/'voyages_2026_synthetic.csv')
print('Raíz:',ROOT.name,'| viajes:',len(voyages),'| periodo:',voyages.date.min(),voyages.date.max())

Raíz: levante_ferries_sustainability | viajes: 4860 | periodo: 2026-01-01 2026-08-31


## Propósito
Inventario de CO₂, CH₄ y N₂O tank to wake; CO₂e con GWP fijos del modelo. Ciclo de vida well to wake separado. ETS: el bio combustible solo obtiene CO₂ biogénico cero en este ejemplo condicionado a prueba de sostenibilidad y trazabilidad. No es un informe MRV verificable.

In [2]:
f=assess(voyages)
print('TTW t CO₂e: %.1f | WTW t CO₂e: %.1f | combustible t: %.1f'%(f.ttw_co2e_kg.sum()/1000,f.wtw_co2e_kg.sum()/1000,f.fuel_kg.sum()/1000))
print('CO₂ t: %.1f | CH₄ t: %.2f | N₂O t: %.3f'%(f.co2_kg.sum()/1000,f.ch4_kg.sum()/1000,f.n2o_kg.sum()/1000))

TTW t CO₂e: 40355.7 | WTW t CO₂e: 48834.4 | combustible t: 12128.5
CO₂ t: 35644.8 | CH₄ t: 151.86 | N₂O t: 1.732


In [3]:
assert abs(f.ttw_co2e_kg.sum()-(f.co2_kg+28*f.ch4_kg+265*f.n2o_kg).sum())<1e-6
print('kg CO₂e/milla: %.2f; kg/pasajero: %.2f; kg/metro lineal: %.2f'%(f.ttw_co2e_kg.sum()/f.distance_nm.sum(),f.ttw_co2e_kg.sum()/f.passengers.sum(),f.ttw_co2e_kg.sum()/f.cargo_lm.sum()))

kg CO₂e/milla: 81.25; kg/pasajero: 12.85; kg/metro lineal: 8.24


**Denominadores:** al agregar se dividen sumas; el indicador por pasajero y por metro lineal son intensidades paralelas del viaje compartido, no una atribución de emisiones que pueda sumarse.